# 03 Serving and Monitoring

**Purpose:** Deploy a registered model to a serving endpoint (Databricks Model Serving or a local alternative), run a canary test with a small percentage of traffic, validate outputs, and demonstrate rollback and monitoring triggers.

**Run order:** Execute sequentially on a Databricks cluster. Some cells are conceptual and show REST API examples — replace placeholders with your workspace values and secrets.

## 1 Install / verify dependencies
Run this cell if the cluster session does not already have the required packages.

In [ ]:
%pip install -q mlflow requests flask gunicorn pandas scikit-learn

import importlib
for pkg in ["mlflow","requests","flask","pandas","sklearn"]:
    try:
        m = importlib.import_module(pkg)
        print(pkg, getattr(m, "__version__", "unknown"))
    except Exception as e:
        print(pkg, "not available:", e)

## 2 Quick check: registered model exists
Confirm the model you want to serve is registered and has a staged version (e.g., `Staging` or `Production`).

In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
registered_model_name = "demo_rf_model"  # adjust if different

models = [m.name for m in client.list_registered_models()]
print("Registered models:", models)

try:
    versions = client.get_latest_versions(registered_model_name)
    for v in versions:
        print(f"version={v.version}, stage={v.current_stage}, run_id={v.run_id}")
except Exception as e:
    print("Could not list versions for", registered_model_name, e)

## 3 Option A — Databricks Model Serving (conceptual)
If your workspace has Databricks Model Serving enabled, create an endpoint in the UI or via the REST API. Below is a conceptual example using the Databricks REST API to create or update an endpoint and route traffic. Replace placeholders with your `DATABRICKS_HOST` and `DATABRICKS_TOKEN` stored in secrets.

In [ ]:
# Example: create/update a Databricks Model Serving endpoint (conceptual)
import os, json, requests

try:
    DATABRICKS_HOST = dbutils.secrets.get("lab-secrets", "DATABRICKS_HOST")
    DATABRICKS_TOKEN = dbutils.secrets.get("lab-secrets", "DATABRICKS_TOKEN")
except Exception:
    DATABRICKS_HOST = None
    DATABRICKS_TOKEN = None

if DATABRICKS_HOST and DATABRICKS_TOKEN:
    # Example payload to create an endpoint (fields vary by API version)
    endpoint_payload = {
        "name": "demo_rf_endpoint",
        "config": {
            "served_models": [
                {"model_name": registered_model_name, "model_version": "Staging", "workload_size": "Small"}
            ]
        }
    }
    headers = {"Authorization": f"Bearer {DATABRICKS_TOKEN}", "Content-Type": "application/json"}
    # POST to the appropriate Databricks Model Serving endpoint creation API
    print("Databricks host and token found. Use REST API to create/update endpoint.")
else:
    print("Databricks secrets not available in this environment. Skipping REST call example.")


## 4 Option B — Local Flask serving (lab alternative)
If Model Serving is not available, run a local Flask app that loads the model artifact from MLflow and serves predictions. This is useful for canary simulation in a controlled environment.

In [ ]:
%%bash
cat > scripts/serve_local.py <<'PY'
from flask import Flask, request, jsonify
import mlflow.pyfunc
import pandas as pd
import os

app = Flask(__name__)

# Load model from registry Staging (adjust name)
MODEL_URI = os.environ.get('LOCAL_MODEL_URI', 'models:/demo_rf_model/Staging')
print('Loading model from', MODEL_URI)
model = mlflow.pyfunc.load_model(MODEL_URI)

@app.route('/predict', methods=['POST'])
def predict():
    payload = request.get_json(force=True)
    data = payload.get('data')
    df = pd.DataFrame(data)
    preds = model.predict(df)
    return jsonify({'predictions': preds.tolist()})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
PY
echo 'Local serve script written to scripts/serve_local.py'


Run the Flask server in a separate terminal or background process if you want to test locally (not recommended on shared clusters). Example (Linux/macOS):
```
export LOCAL_MODEL_URI=models:/demo_rf_model/Staging
python scripts/serve_local.py
```
Then POST to `http://<host>:5000/predict` with JSON `{"data": [[...], [...]]}`.

## 5 Canary traffic simulation
Simulate sending a small percentage of requests to the canary (new model) and the rest to the baseline. For the lab we simulate two endpoints: `baseline_url` and `canary_url`. Replace with real endpoints when available.

In [ ]:
import random, requests, json
import numpy as np
import pandas as pd

# Example test payloads using X_test from previous notebook if available
try:
    sample_X = X_test.reset_index(drop=True)
    sample_rows = sample_X.head(100).to_dict(orient='records')
except Exception:
    # fallback synthetic payloads
    sample_rows = [ { 'f1': random.random()*10, 'f2': random.random()*5, 'avg_f1': random.random()*10, 'avg_f2': random.random()*5, 'num_events': random.randint(1,30) } for _ in range(100) ]

# Replace these with real URLs if you have endpoints
baseline_url = 'http://localhost:5000/predict'  # baseline service
canary_url = 'http://localhost:5001/predict'    # canary service (simulate separate process)

def send_request(url, payload):
    try:
        resp = requests.post(url, json={'data': payload}, timeout=5)
        return resp.json().get('predictions')
    except Exception as e:
        return {'error': str(e)}

# Simulate 100 requests with 5% to canary
canary_pct = 0.05
results = {'baseline': [], 'canary': [], 'errors': []}
for i in range(len(sample_rows)):
    payload = [sample_rows[i]]
    if random.random() < canary_pct:
        r = send_request(canary_url, payload)
        if isinstance(r, dict) and 'error' in r:
            results['errors'].append(('canary', r['error']))
        else:
            results['canary'].append(r[0])
    else:
        r = send_request(baseline_url, payload)
        if isinstance(r, dict) and 'error' in r:
            results['errors'].append(('baseline', r['error']))
        else:
            results['baseline'].append(r[0])

print('Canary requests:', len(results['canary']), 'Baseline requests:', len(results['baseline']), 'Errors:', len(results['errors']))


## 6 Canary validation and rollback logic
Compare simple metrics (mean prediction, distribution, or accuracy if labels available). If the canary deviates beyond thresholds, perform rollback by updating the serving configuration or routing rules.

In [ ]:
import numpy as np

def safe_mean(lst):
    return float(np.mean(lst)) if lst else None

mean_baseline = safe_mean(results['baseline'])
mean_canary = safe_mean(results['canary'])
print('mean_baseline:', mean_baseline, 'mean_canary:', mean_canary)

# Example threshold logic
threshold = 0.2  # example absolute difference threshold
if mean_canary is None:
    print('No canary traffic to evaluate; consider increasing canary_pct')
else:
    delta = abs(mean_baseline - mean_canary)
    print('delta:', delta)
    if delta > threshold:
        print('Canary failed threshold. Initiate rollback actions (conceptual).')
        
        # Conceptual rollback: call serving API to revert traffic or remove canary
        # Example (Databricks REST): PATCH endpoint to set served_models back to baseline only
        # resp = requests.patch('<serving-api-url>', headers=headers, json=rollback_payload)
        # assert resp.status_code == 200
    else:
        print('Canary passed. Proceed with gradual rollout.')


## 7 Monitoring: log serving metrics to a Delta table
Record request-level metrics (latency, success, prediction distribution) to a Delta table for downstream alerting and dashboards.

In [ ]:
from pyspark.sql import Row
from datetime import datetime

rows = []
now = datetime.utcnow()
for p in results['baseline']:
    rows.append(Row(endpoint='baseline', ts=now, prediction=float(p), tag='baseline'))
for p in results['canary']:
    rows.append(Row(endpoint='canary', ts=now, prediction=float(p), tag='canary'))
for e in results['errors']:
    rows.append(Row(endpoint=e[0], ts=now, prediction=None, tag='error'))

if rows:
    df = spark.createDataFrame(rows)
    df.write.format('delta').mode('append').saveAsTable('monitoring.serving_metrics')
    print('Wrote serving metrics rows:', len(rows))
else:
    print('No metrics to write')

## 8 Alerting and automated retrain trigger (conceptual)
Create a Databricks SQL query or scheduled job that checks `monitoring.serving_metrics` for anomalies (e.g., prediction distribution shift, high error rate). Configure an alert to call a webhook that triggers a Databricks Job to run a retraining notebook.

In [ ]:
# Example: simple alert check (run as a scheduled notebook or job)
from pyspark.sql.functions import col

metrics_df = spark.table('monitoring.serving_metrics')
total = metrics_df.count()
errors = metrics_df.filter(col('tag') == 'error').count()
error_rate = errors / total if total > 0 else 0.0
print('total:', total, 'errors:', errors, 'error_rate:', error_rate)

# If error_rate exceeds threshold, trigger retrain job via REST API
if error_rate > 0.1:
    print('High error rate detected. Trigger retrain job (conceptual).')
    # Use Databricks Jobs API to run a retrain job (requires DATABRICKS_HOST and DATABRICKS_TOKEN)
else:
    print('No alert conditions met')

## 9 Tests and assertions
Basic checks to ensure canary traffic was sent and metrics recorded.

In [ ]:
# Test 1: Canary traffic was sent (if canary_pct > 0)
assert isinstance(results, dict), 'results must be a dict'
assert 'baseline' in results and 'canary' in results, 'results missing keys'
print('Baseline count:', len(results['baseline']), 'Canary count:', len(results['canary']))

# Test 2: Metrics table exists and has recent rows
try:
    recent = spark.table('monitoring.serving_metrics').orderBy(col('ts').desc()).limit(5).collect()
    print('Recent metrics rows:', len(recent))
except Exception as e:
    print('Could not read monitoring.serving_metrics:', e)


## 10 Notes and production considerations
- **Traffic routing:** Use your serving platform's traffic splitting features rather than client-side routing for production canaries.  
- **Observability:** Capture latency, error codes, input schema hashes, and prediction distributions.  
- **Automated rollback:** Implement automated rollback when multiple signals (accuracy drop, latency spike, error rate) cross thresholds.  
- **Security:** Protect endpoints with authentication and rate limits.  
- **Testing:** Run canary for a sustained window and use statistical tests (KS, JS divergence) for distribution comparisons.  

Next: `04-Monitoring-Retraining.ipynb` implements drift detection and automated retraining triggers.